# 9. Run 1: ten-passage Gemini at 0.50

Provisional user-preferred recommendation; improves overlap but fails abstention guardrails.

This is a recorded full-train comparison, not a new evaluation.

In [1]:
from pathlib import Path
import json,sys
import pandas as pd
ROOT=next(p for p in [Path.cwd().resolve(),*Path.cwd().resolve().parents] if (p/'src/retrieval.py').exists())
sys.path.insert(0,str(ROOT/'src'))
sys.path.insert(0,str(ROOT/'scripts'))
from presentation_review import selection,cached_demo
model=selection(9)
model['config_snapshot']

{'name': 'llm_hybrid_gemini_broad10',
 'provider': 'gemini',
 'task': 'hybrid_recall_recovery',
 'model': 'gemini-3.5-flash-lite',
 'model_revision': 'API alias; actual modelVersion recorded per response',
 'runtime': 'gemini-rest-v1beta',
 'prompt_file': 'prompts/summarise_complete_clauses_v2.txt',
 'temperature': 0,
 'seed': None,
 'max_output_tokens': 3072,
 'context_length': 1048576,
 'thinking_level': 'minimal',
 'candidate_k': 10,
 'retrieval_config': 'configs/hybrid_recall_broad10.json',
 'max_evidence_quotes': 5,
 'max_summary_words': 45,
 'invalid_output_policy': 'return_original_candidates',
 'expose_in_demo': True,
 'stage': 'broad10'}

In [2]:
m=model['train_metrics']
table=pd.DataFrame(m['retrieval'])[['k','n_queries','precision','recall','hit']]
table

,k,n_queries,precision,recall,hit
0,1,140,0.659853,0.474756,0.692857
1,3,140,0.657558,0.586374,0.778571
2,5,140,0.656978,0.613776,0.800000
3,10,140,0.656235,0.652114,0.821429


In [3]:
print('Correct negative abstention:',m['negative_counts'])
print('False answerable abstention:',m['false_abstention'])
print('Cached latency:',m['latency_ms'])

Correct negative abstention: {'n': 23, 'correct': 17}
False answerable abstention: 0.12142857142857143
Cached latency: {'median': 12.37290003336966, 'p95': 16.2987899268046}


## Cached example

The question comes from recorded train predictions. Config settings and original source offsets remain visible; the model cannot call a hosted provider here.

In [4]:
demo=cached_demo(model)
print(demo['query'])
print(demo.get('summary') or demo['message'])
pd.DataFrame(demo['hits'])[['file_path','start','end','text']]

Consider the Amendment No. 3 to Wireless Content License Agreement between Fox Mobile Entertainment, Inc. and Glu Mobile Inc.; What is the expiration date of this contract?
The agreement expires on December 31, 2006, while rights and licenses for specific properties continue until March 31, 2008.


,file_path,start,end,text
0,cuad/GluMobileInc_20070319_S-1A_EX-10.09_43663...,591,989,"""TERM: The rights granted hereunder shall be e..."


Full code walkthrough: ../experiments/20_gemini_broad10.ipynb. Historical acceptance and current preference are separate; see REPORT.md for limitations.